# Temporal-CausalPFN — Training & Evaluation

This notebook reproduces the experiments from the paper:
> *"Temporal Encoder for Irregular Clinical Time-Series: Adapting Causal PFNs via LoRA and Continuous-Time Positional Encodings"*

## Workflow
1. **Train** temporal encoder + LoRA on synthetic DGP data (`scripts/train.py`)
2. **Evaluate** on synthetic fair comparison (`scripts/evaluate_fair_comparison.py`)
3. **Transfer** to real MIMIC-IV clinical trajectories (`scripts/evaluate_mimic.py`)

## Data
- **Synthetic**: Ornstein-Uhlenbeck process with competing-event observation model
- **MIMIC-IV Demo v2.2**: 100 patients, open-access from [PhysioNet](https://physionet.org/content/mimic-iv-demo/2.2/)
  - Semi-synthetic evaluation: real trajectories + simulated outcomes from known SCM

## Before Running
1. Runtime → Change runtime type → **T4 GPU** (or A100 on Pro+)
2. Clone or upload the repo containing `src/`, `data/`, `scripts/`, and `mimic_iv/`
3. Add your TabPFN token to **Colab Secrets** (see Section 1)

## 0. Setup

In [2]:
# Verify GPU is available
import torch
assert torch.cuda.is_available(), "No GPU detected — change runtime type to GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

AssertionError: No GPU detected — change runtime type to GPU

In [ ]:
# Install dependencies
!pip install -q causalpfn tabpfn-client

In [ ]:
# Clone repo or upload project files
# Option A: Clone from GitHub
!git clone https://github.com/darrenjian/Temporal-CausalPFN.git /content/Temporal-CausalPFN

# Option B: Upload zip (uncomment if needed)
# from google.colab import files
# print("Upload Temporal-CausalPFN.zip")
# uploaded = files.upload()
# !unzip -qo Temporal-CausalPFN.zip -d /content/Temporal-CausalPFN

# Download MIMIC-IV demo from https://physionet.org/content/mimic-iv-demo/2.2/
# and place in /content/Temporal-CausalPFN/mimic_iv/

In [ ]:
# # Option B: Mount Google Drive instead (uncomment if preferred)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/final_project /content/final_project

In [ ]:
%cd /content/Temporal-CausalPFN
!ls -la
!ls scripts/

## 1. Set TabPFN Token (Secure Method)

TabPFN Cloud API requires authentication:
1. Go to https://www.tabpfn.com and register/log in
2. Get your API token from the dashboard
3. **Add it to Colab Secrets** (click 🔑 key icon in left sidebar):
   - Name: `TABPFN_TOKEN`
   - Value: your token
   - Enable notebook access

This keeps your token secure and out of version control.

In [ ]:
import os
from google.colab import userdata

# Get token from Colab Secrets (secure - not in code)
try:
    API_TOKEN = userdata.get('TABPFN_TOKEN')
    os.environ["TABPFN_TOKEN"] = API_TOKEN
except Exception as e:
    print(f"Could not get TABPFN_TOKEN from Colab Secrets: {e}")
    print("Add your token to Colab Secrets (🔑 icon in sidebar)")
    API_TOKEN = None

# Verify TabPFN client works
if API_TOKEN:
    try:
        import tabpfn_client
        from tabpfn_client import TabPFNRegressor
        tabpfn_client.set_access_token(API_TOKEN)
        
        # Quick test
        reg = TabPFNRegressor()
        print("TabPFN client authenticated and ready!")
    except ImportError:
        print("tabpfn_client not installed. Run: pip install tabpfn-client")
    except Exception as e:
        print(f"TabPFN authentication failed: {e}")

## 2. Smoke Test

Quick 5-step training to verify everything imports and runs before the full sweep.

In [ ]:
!python scripts/train.py --config strong_temporal --n_steps 5 --batch_size 32 \
    --checkpoint_dir /tmp/smoke_test --device cuda

## 3. Train All Configs × Seeds

Trains the encoder + LoRA rank 32 on 4 main configs × 3 seeds.
Each run is ~20 min on A100, ~40-60 min on T4.

**Checkpoints auto-resume** — if Colab disconnects, re-run and it picks up where it left off.

The paper's Table 3 (fair comparison) uses all configs. The MIMIC evaluation uses the same
checkpoints to test transfer to real clinical data. Training `strong_temporal` across 3 seeds
is the minimum required.

In [ ]:
import time

CONFIGS = ["tabular_sufficient", "weak_temporal", "strong_temporal", "asymmetric"]
SEEDS = [42, 123, 999]
N_STEPS = 3000

total_runs = len(CONFIGS) * len(SEEDS)
completed = 0

for config in CONFIGS:
    for seed in SEEDS:
        ckpt_dir = f"./checkpoints/{config}/{seed}"
        completed += 1
        print(f"\n{'='*65}")
        print(f"Run {completed}/{total_runs}: config={config}, seed={seed}")
        print(f"{'='*65}")
        !python scripts/train.py \
            --config {config} \
            --n_steps {N_STEPS} \
            --batch_size 64 \
            --checkpoint_dir {ckpt_dir} \
            --seed {seed} \
            --device cuda

print(f"\nAll {total_runs} training runs complete.")

In [ ]:
# Save checkpoints to Drive (optional, protects against disconnect)
# !cp -r ./checkpoints /content/drive/MyDrive/checkpoints

## 4. MIMIC Semi-Synthetic Evaluation

Runs the 4-condition fair comparison on real MIMIC-IV clinical trajectories.
Results populate the MIMIC section of Table 3 and Table 4 (PEHE) in `paper.tex`.

**Key test**: Does the encoder trained on *synthetic* OU-process data produce
useful representations for *real* clinical observation patterns?

In [ ]:
!python scripts/evaluate_mimic.py \
    --checkpoint_dir ./checkpoints \
    --mimic_dir ./mimic_iv \
    --output_dir ./results_mimic \
    --device cuda \
    --n_eval_patients 500

## 5. Synthetic Fair Comparison Evaluation

Runs the fair comparison on synthetic data (V1 and V2 sections of Table 3).
Verifies temporal embeddings improve both CausalPFN and TabPFN-X.

In [ ]:
!python scripts/evaluate_fair_comparison.py \
    --checkpoint_dir ./checkpoints \
    --output_dir ./results_fair \
    --device cuda

## 6. Format Results for Paper

Reads the MIMIC evaluation JSON and prints LaTeX-ready table rows.

In [ ]:
import json
import glob

# Find the most recent MIMIC results JSON
result_files = sorted(glob.glob("./results_mimic/mimic_eval_results_*.json"))
if not result_files:
    print("No MIMIC results found. Run Section 4 first.")
else:
    with open(result_files[-1]) as f:
        results = json.load(f)
    print(f"Loaded: {result_files[-1]}\n")

    # Configs matching paper.tex Table 3 (MIMIC section)
    CONFIGS = [
        "tabular_sufficient",
        "weak_temporal",
        "strong_temporal",
        "asymmetric",
        "strong_temporal_trend",
        "trend_only",
    ]
    CONDS = [
        "causalpfn_static",
        "tabpfn_x_static",
        "causalpfn_temporal",
        "tabpfn_x_temporal",
    ]

    def fmt_ate(r):
        ate = r.get("ate", {})
        if ate.get("mean") is None:
            return "---"
        return f"{ate['mean']:.1f}$\\pm${ate['std']:.1f}"

    # Print ATE table (Table 3 MIMIC section)
    print("=" * 80)
    print("TABLE 3 (MIMIC): ATE Relative Error (%) — paste into paper.tex")
    print("=" * 80)
    print()
    for config in CONFIGS:
        r = results.get(config, {})
        cells = []
        for cond in CONDS:
            cells.append(fmt_ate(r.get(cond, {})))
        config_tex = config.replace("_", "\\_")
        print(f"{config_tex:<30} & {' & '.join(cells)} \\\\")

    # Print PEHE for Table 4
    print()
    print("=" * 80)
    print("TABLE 4 (MIMIC PEHE)")
    print("=" * 80)
    print()
    for config in CONFIGS:
        r = results.get(config, {})
        cells = []
        for cond in CONDS:
            p = r.get(cond, {}).get("pehe", {})
            if p.get("mean") is None:
                cells.append("---")
            else:
                cells.append(f"{p['mean']:.3f}$\\pm${p['std']:.3f}")
        config_tex = config.replace("_", "\\_")
        print(f"{config_tex:<30} & {' & '.join(cells)} \\\\")

    # Print raw summary
    print()
    print("=" * 80)
    print("Raw summary (all configs, all conditions)")
    print("=" * 80)
    for config, conds in results.items():
        print(f"\n{config}:")
        for cond, metrics in conds.items():
            p = metrics.get("pehe", {})
            a = metrics.get("ate", {})
            if p.get("mean") is not None:
                print(f"  {cond:<25} PEHE={p['mean']:.4f}±{p['std']:.4f}  ATE_err={a['mean']:.1f}%±{a['std']:.1f}%")
            else:
                print(f"  {cond:<25} NO RESULTS")

## 7. Download Results

Download all result files and checkpoints for local use.

In [ ]:
# Package results for download
!tar czf /content/results_all.tar.gz ./results_mimic ./results_fair

from google.colab import files
files.download('/content/results_all.tar.gz')
print("Downloaded results_all.tar.gz")

In [ ]:
# # Optional: download checkpoints too (large)
# !tar czf /content/checkpoints.tar.gz ./checkpoints
# files.download('/content/checkpoints.tar.gz')